### Raw Data Cleaning

In [1]:
import pandas as pd

In [2]:
locations_df = pd.read_csv("../data/locations.csv")
providers_df = pd.read_csv("../data/providers.csv")
payers_df = pd.read_csv("../data/payers.csv")
patients_df = pd.read_csv("../data/patients.csv")
calls_df = pd.read_csv("../data/calls.csv")
appointments_df = pd.read_csv("../data/appointments.csv")

for name, df in [("locations", locations_df), ("providers", providers_df),
                  ("payers", payers_df), ("patients", patients_df),
                  ("calls", calls_df), ("appointments", appointments_df)]:
    print(name, df.shape)

locations (4, 4)
providers (8, 5)
payers (6, 3)
patients (9439, 4)
calls (28493, 6)
appointments (64534, 12)


In [3]:
def strip_text_columns(df):
    text_cols = df.select_dtypes(include=['object', 'string']).columns
    for col in text_cols:
        df[col] = df[col].str.strip()
        
    return df

locations_df = strip_text_columns(locations_df)
payers_df = strip_text_columns(payers_df)
patients_df = strip_text_columns(patients_df)
calls_df = strip_text_columns(calls_df)
providers_df = strip_text_columns(providers_df)
appointments_df = strip_text_columns(appointments_df)

def strip_column_titles(df):
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    return df

locations_df = strip_column_titles(locations_df)
payers_df = strip_column_titles(payers_df)
patients_df = strip_column_titles(patients_df)
calls_df = strip_column_titles(calls_df)
providers_df = strip_column_titles(providers_df)
appointments_df = strip_column_titles(appointments_df)

### Appointments

In [4]:
appointments_df.head()

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
0,A1,2023-08-01,2023-07-23,P1,L1,PT1,PY1,Follow-up,False,Completed,92.03,0.93
1,A2,2023-08-01,2023-07-27,P1,L1,PT1,PY1,Physical Therapy,False,Cancelled,0.00,0.00
2,A3,2023-08-01,2023-08-01,P1,L1,PT1,PY1,Post-Op Check,False,Completed,75.98,0.57
3,A4,2023-08-01,2023-07-26,P1,L1,PT1,PY1,Follow-up,False,Completed,130.28,0.68
4,A5,2023-08-01,2023-07-30,P1,L1,PT1,PY1,Follow-up,False,Completed,137.98,1.03


In [5]:
appointments_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64534 entries, 0 to 64533
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   appointment_id    64534 non-null  object 
 1   date              64534 non-null  object 
 2   booked_date       64534 non-null  object 
 3   provider_id       64534 non-null  object 
 4   location_id       64534 non-null  object 
 5   patient_id        64534 non-null  object 
 6   payer_id          64534 non-null  object 
 7   appointment_type  64534 non-null  object 
 8   is_new_patient    64534 non-null  bool   
 9   status            64534 non-null  object 
 10  revenue           64012 non-null  float64
 11  rvu               64534 non-null  float64
dtypes: bool(1), float64(2), object(9)
memory usage: 5.5+ MB


In [6]:
# Convert dates to datetime format
appointments_df['date'] = pd.to_datetime(appointments_df['date'], errors='coerce', format='%Y-%m-%d')
appointments_df['booked_date'] = pd.to_datetime(appointments_df['booked_date'], errors='coerce', format='%Y-%m-%d')

In [7]:
# Clean column names by stripping whitespace and converting to lowercase
# appointments_df.columns = appointments_df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
# appointments_df.columns.tolist()

# Redundant now, swap to bulk processing for all dataframes

In [8]:
appointments_df.isna().sum()

appointment_id        0
date                  0
booked_date           0
provider_id           0
location_id           0
patient_id            0
payer_id              0
appointment_type      0
is_new_patient        0
status                0
revenue             522
rvu                   0
dtype: int64

In [ ]:
# Find rows where revenue is null and status is Completed
# Meaning that the appointment was completed but revenue was not posted yet, might require followup if lost
blank_revenue_df = appointments_df[(appointments_df['revenue'].isna()) & (appointments_df['status'] == 'Completed')]
blank_revenue_df.shape

(522, 12)

In [ ]:
# No rows where revenue is 0 and status is Completed, so no need to follow up on those
appointments_df[(appointments_df['revenue'] == 0) & (appointments_df['status'] == 'Completed')]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [ ]:
# Impossible dates, turn them into NaT but keep records for safekeeping rather than dropping completely
bad_dates = appointments_df['booked_date'] > appointments_df['date']
appointments_df.loc[bad_dates, 'booked_date'] = pd.NaT

booked_date
2026-07-09    111
2026-06-23    107
2025-10-31    105
2026-07-02    105
2026-02-20    104
             ... 
2023-07-15      2
2023-07-12      2
2023-07-10      1
2023-07-11      1
2023-07-06      1
Name: count, Length: 1119, dtype: int64

In [61]:
appointments_df['booked_date'].value_counts()

booked_date
2026-07-09    111
2026-06-23    107
2025-10-31    105
2026-07-02    102
2026-02-20    100
             ... 
2023-07-15      2
2023-07-12      2
2023-07-10      1
2023-07-06      1
2023-07-11      1
Name: count, Length: 1119, dtype: int64

In [ ]:
# Primary key validation
appointments_df['appointment_id'].nunique() == len(appointments_df)

False

In [13]:
# Drop duplicate rows based on the 'appointment_id' column
appointments_df.drop_duplicates(subset=['appointment_id'], keep='first', inplace=True)
appointments_df.shape # 64534 -> 64213

(64213, 12)

In [ ]:
# Fine now
appointments_df['appointment_id'].nunique() == len(appointments_df)

True

In [ ]:
# Provider validation, find appointments with provider_id that doesn't exist in providers_df
set(providers_df['provider_id'])
provider_error_appointments_df = appointments_df[~appointments_df['provider_id'].isin(set(providers_df['provider_id']))]
provider_error_appointments_df

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
689,A690,2023-08-16,2023-08-10,P99,L2,PT67,PY3,Post-Op Check,False,Completed,87.12,0.76
1056,A1057,2023-08-24,2023-07-31,P99,L3,PT156,PY3,New Patient Consult,True,No-Show,0.00,0.00
2262,A2263,2023-09-21,2023-09-18,P99,L1,PT227,PY4,Physical Therapy,False,Completed,123.63,1.32
2883,A2884,2023-10-05,2023-10-05,P99,L1,PT33,PY5,Physical Therapy,False,Completed,124.15,1.40
3138,A3139,2023-10-10,2023-10-02,P99,L3,PT250,PY2,Follow-up,False,Completed,89.65,0.83
...,...,...,...,...,...,...,...,...,...,...,...,...
61217,A61218,2026-06-24,2026-06-12,P99,L1,PT4486,PY1,Follow-up,False,Completed,125.67,0.83
61387,A61388,2026-06-25,2026-06-24,P99,L4,PT3169,PY3,Follow-up,False,Completed,79.03,1.01
62024,A62025,2026-07-03,2026-06-18,P99,L1,PT5714,PY6,Follow-up,False,Completed,119.12,1.08
63150,A63151,2026-07-17,2026-07-06,P99,L2,PT4338,PY1,Follow-up,False,Completed,89.56,0.86


In [15]:
# P99 doesn't exist in providers_df, but it does exist in appointments_df. This is likely a data quality issue that needs to be addressed.
set(appointments_df['provider_id'])

{'P1', 'P2', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P99'}

In [ ]:
# Some provider_ids in appointments_df do not exist in providers_df. Replace those with 'UNK' to indicate unknown provider.
invalid_provider_mask = ~appointments_df['provider_id'].isin(set(providers_df['provider_id']))
appointments_df.loc[invalid_provider_mask, 'provider_id'] = 'UNK'

appointments_df[appointments_df['provider_id'] == 'UNK']

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu
689,A690,2023-08-16,2023-08-10,UNK,L2,PT67,PY3,Post-Op Check,False,Completed,87.12,0.76
1056,A1057,2023-08-24,2023-07-31,UNK,L3,PT156,PY3,New Patient Consult,True,No-Show,0.00,0.00
2262,A2263,2023-09-21,2023-09-18,UNK,L1,PT227,PY4,Physical Therapy,False,Completed,123.63,1.32
2883,A2884,2023-10-05,2023-10-05,UNK,L1,PT33,PY5,Physical Therapy,False,Completed,124.15,1.40
3138,A3139,2023-10-10,2023-10-02,UNK,L3,PT250,PY2,Follow-up,False,Completed,89.65,0.83
...,...,...,...,...,...,...,...,...,...,...,...,...
61217,A61218,2026-06-24,2026-06-12,UNK,L1,PT4486,PY1,Follow-up,False,Completed,125.67,0.83
61387,A61388,2026-06-25,2026-06-24,UNK,L4,PT3169,PY3,Follow-up,False,Completed,79.03,1.01
62024,A62025,2026-07-03,2026-06-18,UNK,L1,PT5714,PY6,Follow-up,False,Completed,119.12,1.08
63150,A63151,2026-07-17,2026-07-06,UNK,L2,PT4338,PY1,Follow-up,False,Completed,89.56,0.86


In [17]:
appointments_df[~appointments_df['location_id'].isin(set(locations_df['location_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [18]:
appointments_df[~appointments_df['patient_id'].isin(set(patients_df['patient_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [19]:
appointments_df[~appointments_df['payer_id'].isin(set(payers_df['payer_id']))]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [20]:
appointments_df.shape

(64213, 12)

In [ ]:
# Check for negative revenue values
appointments_df[appointments_df['revenue'] < 0]['revenue'].sum()

np.float64(0.0)

In [ ]:
# Check for negative rvu values
appointments_df[appointments_df['rvu'] < 0]['rvu'].sum()

np.float64(0.0)

In [23]:
appointments_df['status'].value_counts()

status
Completed    51943
No-Show       9160
Cancelled     3110
Name: count, dtype: int64

In [ ]:
# Revenue should only ever be present on Completed appointments
# These are basically appointments that never happened (canceled, no-show, etc.) so the revenue are just 0.00
appointments_df[(appointments_df['status'] != 'Completed') & (appointments_df['revenue'].notna())]['revenue'].unique()

array([0.])

In [25]:
appointments_df['appointment_type'].value_counts()

appointment_type
Follow-up              28908
Physical Therapy       12890
Post-Op Check           9818
New Patient Consult     9438
Injection/Procedure     3159
Name: count, dtype: int64

In [26]:
# Check if the number of new patient consultations matches the number of new patients
len(appointments_df[appointments_df['appointment_type'] == 'New Patient Consult']) == len(appointments_df[appointments_df['is_new_patient'] == True])

True

In [ ]:
# Checks if there's any non-completed appointments that have revenue greater than 0, which should not happen
appointments_df[(appointments_df['status'] != 'Completed') & (appointments_df['revenue'] > 0)]

,appointment_id,date,booked_date,provider_id,location_id,patient_id,payer_id,appointment_type,is_new_patient,status,revenue,rvu


In [ ]:
# Makes sure that the revenue values are reasonable
appointments_df['revenue'].describe()

count    63695.000000
mean       115.315204
std         98.559891
min          0.000000
25%         73.290000
50%        101.020000
75%        134.140000
max        599.880000
Name: revenue, dtype: float64

In [73]:
appointments_df['rvu'].describe()

count    64213.000000
mean         0.980854
std          0.823474
min          0.000000
25%          0.620000
50%          0.860000
75%          1.120000
max          4.500000
Name: rvu, dtype: float64

In [ ]:
# 63695 + 518 = 64213, which is the total number of appointments in the dataframe.
# 518 appointments are waiting on the calculated revenue, but they should all have rvu which works out in this case.
appointments_df['revenue'].isna().sum()

np.int64(518)

### Patients

In [28]:
patients_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9439 entries, 0 to 9438
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   patient_id        9439 non-null   object
 1   first_visit_date  9439 non-null   object
 2   referral_source   9439 non-null   object
 3   payer_id          9439 non-null   object
dtypes: object(4)
memory usage: 295.1+ KB


In [ ]:
# Primary key validation
patients_df['patient_id'].nunique() == len(patients_df)

True

In [30]:
patients_df['first_visit_date'] = pd.to_datetime(patients_df['first_visit_date'], errors='coerce', format='%Y-%m-%d')

In [31]:
patients_df['first_visit_date'].isna().sum()

np.int64(0)

In [32]:
patients_df['first_visit_date'].value_counts()

first_visit_date
2026-03-18    26
2026-02-03    25
2026-05-19    25
2025-08-18    23
2025-12-22    23
              ..
2024-11-09     1
2025-01-16     1
2024-12-14     1
2024-12-07     1
2023-09-16     1
Name: count, Length: 936, dtype: int64

In [33]:
patients_df.head()

,patient_id,first_visit_date,referral_source,payer_id
0,PT1,2023-08-01,Friend/Family,PY1
1,PT2,2023-08-01,Insurance Directory,PY2
2,PT3,2023-08-01,Friend/Family,PY3
3,PT4,2023-08-01,Physician Referral,PY6
4,PT5,2023-08-01,Physician Referral,PY1


In [ ]:
# Inaccurate casing causing values to be counted separately when they are actually the same
patients_df['referral_source'].value_counts()

referral_source
Physician Referral     3425
Self                   1803
Insurance Directory    1466
Online Search          1384
Friend/Family           957
Physician referral      152
physician referral      148
self                     56
SELF                     48
Name: count, dtype: int64

In [ ]:
# Clean and check
patients_df['referral_source'] = patients_df['referral_source'].str.strip().str.title()
patients_df['referral_source'].value_counts()

referral_source
Physician Referral     3725
Self                   1907
Insurance Directory    1466
Online Search          1384
Friend/Family           957
Name: count, dtype: int64

In [37]:
# No errors
invalid_payer_mask = ~patients_df['payer_id'].isin(set(payers_df['payer_id']))
patients_df[invalid_payer_mask]

,patient_id,first_visit_date,referral_source,payer_id


In [38]:
# Nothing dropped
patients_df.drop_duplicates()

,patient_id,first_visit_date,referral_source,payer_id
0,PT1,2023-08-01,Friend/Family,PY1
1,PT2,2023-08-01,Insurance Directory,PY2
2,PT3,2023-08-01,Friend/Family,PY3
3,PT4,2023-08-01,Physician Referral,PY6
4,PT5,2023-08-01,Physician Referral,PY1
...,...,...,...,...
9434,PT9435,2026-07-31,Online Search,PY1
9435,PT9436,2026-07-31,Friend/Family,PY3
9436,PT9437,2026-07-31,Friend/Family,PY1
9437,PT9438,2026-07-31,Insurance Directory,PY3


### Payers

In [39]:
payers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6 entries, 0 to 5
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   payer_id    6 non-null      object
 1   payer_name  6 non-null      object
 2   payer_type  6 non-null      object
dtypes: object(3)
memory usage: 276.0+ bytes


In [40]:
payers_df

,payer_id,payer_name,payer_type
0,PY1,Medicare,Medicare
1,PY2,Medi-Cal,Medicaid
2,PY3,Blue Cross,Commercial
3,PY4,Aetna,Commercial
4,PY5,Humana,Commercial
5,PY6,Self-Pay,Self-Pay


In [ ]:
# Primary key validation
payers_df['payer_id'].nunique() == len(payers_df)

True

In [42]:
payers_df['payer_type'].value_counts()

payer_type
Commercial    3
Medicare      1
Medicaid      1
Self-Pay      1
Name: count, dtype: int64

In [ ]:
# Mostly insurance, but there are some self-pay patients as well
# Name and type matches, ex. Medicare with Medicare
payers_df[['payer_name', 'payer_type']]

,payer_name,payer_type
0,Medicare,Medicare
1,Medi-Cal,Medicaid
2,Blue Cross,Commercial
3,Aetna,Commercial
4,Humana,Commercial
5,Self-Pay,Self-Pay


### Providers

In [44]:
providers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   provider_id          8 non-null      object
 1   provider_name        8 non-null      object
 2   specialty            8 non-null      object
 3   primary_location_id  8 non-null      object
 4   hire_date            8 non-null      object
dtypes: object(5)
memory usage: 452.0+ bytes


In [45]:
providers_df

,provider_id,provider_name,specialty,primary_location_id,hire_date
0,P1,Dr. James Nguyen,Orthopedic Surgery,L1,2022-08-02
1,P2,Dr. Maria Patel,Orthopedic Surgery,L1,2019-08-17
2,P3,Dr. Robert Garcia,Sports Medicine,L1,2019-02-21
3,P4,Dr. Linda Kim,Sports Medicine,L2,2023-02-27
4,P5,Dr. David Rossi,Physical Medicine & Rehab,L2,2020-07-17
5,P6,"Susan Chen, PT",Physical Therapy,L3,2020-05-16
6,P7,Dr. Michael Alvarez,Pain Management,L3,2020-04-02
7,P8,Dr. Karen Bennett,Orthopedic Surgery,L4,2025-10-11


In [46]:
providers_df['hire_date'] = pd.to_datetime(providers_df['hire_date'], errors='coerce', format='%Y-%m-%d')

In [47]:
providers_df['hire_date'].isna().sum()

np.int64(0)

In [ ]:
# Validate that all primary_location_id values in providers_df exist in locations_df
invalid_location_mask = ~providers_df['primary_location_id'].isin(set(locations_df['location_id']))
providers_df[invalid_location_mask]

,provider_id,provider_name,specialty,primary_location_id,hire_date


In [ ]:
# Primary key validation
providers_df['provider_id'].nunique() == len(providers_df)

True

In [50]:
providers_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 5 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   provider_id          8 non-null      object        
 1   provider_name        8 non-null      object        
 2   specialty            8 non-null      object        
 3   primary_location_id  8 non-null      object        
 4   hire_date            8 non-null      datetime64[ns]
dtypes: datetime64[ns](1), object(4)
memory usage: 452.0+ bytes


### Locations

In [51]:
locations_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4 entries, 0 to 3
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   location_id    4 non-null      object
 1   location_name  4 non-null      object
 2   city           4 non-null      object
 3   state          4 non-null      object
dtypes: object(4)
memory usage: 260.0+ bytes


In [52]:
locations_df.head()

,location_id,location_name,city,state
0,L1,Santa Clarita Office,Santa Clarita,CA
1,L2,Valencia Office,Valencia,CA
2,L3,Burbank Office,Burbank,CA
3,L4,Glendale Office,Glendale,CA


In [66]:
# Primary key validation
locations_df['location_id'].nunique() == len(locations_df)

True

In [ ]:
# Small sample of locations_df, but all states should be CA. This is just a check for good practice.
len(locations_df[locations_df['state'] == 'CA']) == len(locations_df)

True

In [70]:
locations_df['city'].value_counts()

city
Santa Clarita    1
Valencia         1
Burbank          1
Glendale         1
Name: count, dtype: int64

In [71]:
locations_df['location_name'].value_counts()

location_name
Santa Clarita Office    1
Valencia Office         1
Burbank Office          1
Glendale Office         1
Name: count, dtype: int64

### Calls

In [53]:
calls_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28493 entries, 0 to 28492
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   call_id          28493 non-null  object
 1   date             28493 non-null  object
 2   location_id      28493 non-null  object
 3   call_type        28493 non-null  object
 4   outcome          28493 non-null  object
 5   handle_time_sec  28493 non-null  int64 
dtypes: int64(1), object(5)
memory usage: 1.3+ MB


In [54]:
calls_df.head()

,call_id,date,location_id,call_type,outcome,handle_time_sec
0,C1,2023-08-01,L1,General Inquiry,Info Only,261
1,C2,2023-08-01,L1,New Patient Inquiry,Booked,164
2,C3,2023-08-01,L1,Reschedule,Booked,146
3,C4,2023-08-01,L1,Billing Question,Info Only,259
4,C5,2023-08-01,L1,New Patient Inquiry,Booked,459


In [55]:
calls_df['date'] = pd.to_datetime(calls_df['date'], errors='coerce', format='%Y-%m-%d')

In [ ]:
# Validate that all location_id values in calls_df exist in locations_df
invalid_location_mask = ~calls_df['location_id'].isin(set(locations_df['location_id']))
calls_df[invalid_location_mask]

,call_id,date,location_id,call_type,outcome,handle_time_sec


In [ ]:
# Primary key validation
calls_df['call_id'].nunique() == len(calls_df)

True

In [58]:
calls_df['call_type'].value_counts()

call_type
New Patient Inquiry    10042
Reschedule              8630
General Inquiry         5615
Billing Question        4206
Name: count, dtype: int64

In [59]:
calls_df['outcome'].value_counts()

outcome
Booked        13472
Info Only      8701
Not Booked     6320
Name: count, dtype: int64

In [ ]:
# Make sure that values makes sense, ex. handle_time_sec should be positive and not too high
calls_df['handle_time_sec'].describe()

count    28493.000000
mean       262.907486
std        126.172433
min         45.000000
25%        153.000000
50%        263.000000
75%        372.000000
max        480.000000
Name: handle_time_sec, dtype: float64